In [1]:
!pip install -q beir

In [2]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader

url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"

data_path = util.download_and_unzip(url, "data")

corpus, queries, qrels = GenericDataLoader(
    data_folder=data_path
).load(split="train")

print("Documents:", len(corpus))
print("Claims:", len(queries))
print("Qrels:", len(qrels))

/usr/local/lib/python3.13/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


data/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

  0%|          | 0/5183 [00:00<?, ?it/s]

Documents: 5183
Claims: 809
Qrels: 809


In [3]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

doc_ids = list(corpus.keys())

documents = [
    corpus[doc_id]["title"] + " " + corpus[doc_id]["text"]
    for doc_id in doc_ids
]

document_embeddings = model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(document_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/162 [00:00<?, ?it/s]

(5183, 384)


In [4]:
def semantic_retrieve(claim, top_k=5):

    query_embedding = model.encode(
        claim,
        normalize_embeddings=True
    )

    scores = np.dot(document_embeddings, query_embedding)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append({
            "doc_id": doc_ids[idx],
            "title": corpus[doc_ids[idx]]["title"],
            "score": float(scores[idx])
        })

    return results

In [5]:
import numpy as np

def calculate_metrics(retrieved_ids, relevant_ids, k=5):

    retrieved_ids = retrieved_ids[:k]

    if len(relevant_ids) == 0:
        return None

    relevant_retrieved = [
        doc_id for doc_id in retrieved_ids
        if doc_id in relevant_ids
    ]

    precision = len(relevant_retrieved) / k

    recall = (
        len(set(relevant_retrieved)) / len(relevant_ids)
    )

    reciprocal_rank = 0

    for rank, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant_ids:
            reciprocal_rank = 1 / rank
            break

    return {
        "precision": precision,
        "recall": recall,
        "reciprocal_rank": reciprocal_rank
    }

In [6]:
all_metrics = []

for query_id in queries:

    claim = queries[query_id]

    results = semantic_retrieve(claim, top_k=5)

    retrieved_ids = [
        str(result["doc_id"]) for result in results
    ]

    relevant_ids = set(
        str(doc_id) for doc_id in qrels[query_id].keys()
    )

    metrics = calculate_metrics(
        retrieved_ids,
        relevant_ids,
        k=5
    )

    if metrics is not None:
        all_metrics.append(metrics)

mean_precision = np.mean([
    m["precision"] for m in all_metrics
])

mean_recall = np.mean([
    m["recall"] for m in all_metrics
])

mrr = np.mean([
    m["reciprocal_rank"] for m in all_metrics
])

print("Precision@5:", mean_precision)
print("Recall@5:", mean_recall)
print("MRR:", mrr)

Precision@5: 0.16365883807169343
Recall@5: 0.7347342398022251
MRR: 0.6211990111248455


In [8]:
def retrieve_candidates(claim, top_k=20):

    query_embedding = model.encode(
        claim,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding.reshape(1, -1),
        top_k
    )

    candidates = []

    for score, idx in zip(scores[0], indices[0]):

        doc_id = doc_ids[idx]

        candidates.append({
            "doc_id": doc_id,
            "title": corpus[doc_id]["title"],
            "text": corpus[doc_id]["text"],
            "retrieval_score": float(score)
        })

    return candidates


def rerank_documents(claim, candidates, top_k=5):

    pairs = [
        (claim, candidate["title"] + " " + candidate["text"])
        for candidate in candidates
    ]

    scores = reranker.predict(pairs)

    for candidate, score in zip(candidates, scores):
        candidate["rerank_score"] = float(score)

    return sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )[:top_k]


print("Both functions defined successfully!")

Both functions defined successfully!


In [12]:
!pip install -q faiss-cpu

In [13]:
import faiss
import numpy as np

print("FAISS version:", faiss.__version__)

FAISS version: 1.16.0


In [14]:
dimension = document_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    document_embeddings.astype("float32")
)

print("FAISS index created!")
print("Total indexed documents:", index.ntotal)

FAISS index created!
Total indexed documents: 5183


In [16]:
test_candidates = retrieve_candidates(queries["0"], top_k=20)

print("Retrieved candidates:", len(test_candidates))
print(test_candidates[0]["title"])

Retrieved candidates: 20
Complex Tissue and Disease Modeling using hiPSCs.


In [15]:
import faiss
import numpy as np

# Get embedding dimension
dimension = document_embeddings.shape[1]

# Create FAISS index
index = faiss.IndexFlatIP(dimension)

# Add document embeddings
index.add(
    document_embeddings.astype("float32")
)

print("FAISS index created!")
print("Total indexed documents:", index.ntotal)

FAISS index created!
Total indexed documents: 5183


In [18]:
!pip install -q sentence-transformers

In [19]:
from sentence_transformers import CrossEncoder
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device=device
)

print("Cross-Encoder loaded successfully!")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder loaded successfully!


In [20]:
test_score = reranker.predict([
    ("Vitamin D reduces infections",
     "Vitamin D supplementation was associated with reduced respiratory infections.")
])

print("Test score:", test_score)

Test score: [6.8071485]


In [21]:
reranked_metrics = []

for query_id in queries:

    claim = queries[query_id]

    candidates = retrieve_candidates(claim, top_k=20)

    results = rerank_documents(
        claim,
        candidates,
        top_k=5
    )

    retrieved_ids = [
        str(result["doc_id"]) for result in results
    ]

    relevant_ids = set(
        str(doc_id) for doc_id in qrels[query_id].keys()
    )

    metrics = calculate_metrics(
        retrieved_ids,
        relevant_ids,
        k=5
    )

    if metrics is not None:
        reranked_metrics.append(metrics)

print(
    "Reranked Recall@5:",
    np.mean([m["recall"] for m in reranked_metrics])
)

Reranked Recall@5: 0.7710341985990935
